In [5]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
import numpy as np
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestRegressor

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()
    
    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        N * 14 / Descriptors.MolWt(mol),
        NO_single / sum, NO_double / sum,
        NN_single / sum, NN_double / sum
    ]
    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
x_all = np.stack(df["ROMol"].apply(mol_to_feat).tolist(), axis = 0)
y_all = np.array(df["Q(cal/g)"], dtype = np.float64)

rf = RandomForestRegressor(
    n_estimators = 200,
    n_jobs = -1,
    random_state = 114514
)
scores = cross_validate(rf, x_all, y_all, cv = 10, scoring = ["neg_mean_squared_error", "neg_mean_absolute_error", "r2"], n_jobs = -1)
mse = np.abs(scores["test_neg_mean_squared_error"]).mean()
mae = np.abs(scores["test_neg_mean_absolute_error"]).mean()
r2 = scores["test_r2"].mean()
print(mse, mae, r2, sep = '\n')


34824.424579139195
149.99056170573493
0.7568835900044162


In [ ]:
from pathlib import Path
import pandas as pd
from rdkit import Chem
from rdkit.Chem import PandasTools, Descriptors, rdchem, rdMolDescriptors
import numpy as np
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor

def mol_to_feat(mol):
    #找我认为的重要键
    NO_single = NO_double = 0
    NN_single = NN_double = 0
    sum = 0
    for i in mol.GetBonds():
        pre = i.GetBeginAtom().GetSymbol()
        nxt = i.GetEndAtom().GetSymbol()
        link = i.GetBondType()
        if pre == "O":
            pre, nxt = nxt, pre
        if pre == "N" and nxt == "O":
            if link == rdchem.BondType.SINGLE:
                NO_single += 1
            else:
                NO_double += 1
        elif pre == "N" and nxt == "N":
            if link == rdchem.BondType.SINGLE:
                NN_single += 1
            else:
                NN_double += 1
        sum += 1

    #算各种原子的个数
    mqn = rdMolDescriptors.MQNs_(mol)
    N = mqn[7] + mqn[8]
    O = mqn[9] + mqn[10]
    C = mqn[0]
    mol = Chem.AddHs(mol)
    H = mol.GetNumAtoms() - mol.GetNumHeavyAtoms()
    
    feat = [
        (C * 2 + H / 2 - O) * 16 / Descriptors.MolWt(mol),
        N * 14 / Descriptors.MolWt(mol),
        NO_single / sum, NO_double / sum,
        NN_single / sum, NN_double / sum
    ]
    return feat

file = Path.cwd().parent / "data" / "test.xlsx"
df = pd.read_excel(file)
PandasTools.AddMoleculeColumnToFrame(df, "SMILES", "ROMol", False)
df = df.sample(frac = 1, random_state = 1189, ignore_index = True)

x_train = np.stack(df["ROMol"][ : 71].apply(mol_to_feat).tolist(), axis = 0)
x_prac = np.stack(df["ROMol"][71 : ].apply(mol_to_feat).tolist(), axis = 0)
y_train = np.array(df["Q(cal/g)"][ : 71], dtype = np.float64)
y_prac = np.array(df["Q(cal/g)"][71 : ], dtype = np.float64)

rf = RandomForestRegressor(
    n_estimators = 200,
    n_jobs = -1,
    random_state = 323922
)
rf.fit(x_train, y_train)
y_pre = rf.predict(x_prac)

PandasTools.RenderImagesInAllDataFrames(images = True)
df1 = df[71 : ]
df1["predict"] = y_pre
print(r2_score(y_prac, y_pre))
df1
